In [2]:
import numpy, gsw

In [3]:
# ad-hoc implementation
def AOU_estimate(potential_temperature, salinity, density, oxygen):
    # function to compute AOU in umol/kg
    # based on Sarmiento & Gruber (Garcia and Gordon, 1992)
    # all inputs can be either scalar or 1D iterables of the same length
    # potential_temperature in degrees Celsius
    # salinity in situ in PSU
    # density in situ in kg/m3
    # oxygen in situ in umol/kg

    # convert to arrays; None -> nan
    pt = numpy.asarray(potential_temperature, dtype=float)
    sa = numpy.asarray(salinity, dtype=float)
    ro = numpy.asarray(density, dtype=float)
    o2 = numpy.asarray(oxygen, dtype=float)

    scalar = (pt.ndim == sa.ndim == ro.ndim == o2.ndim == 0)

    # force 1D so we only write the algebra once
    pt = numpy.atleast_1d(pt)
    sa = numpy.atleast_1d(sa)
    ro = numpy.atleast_1d(ro)
    o2 = numpy.atleast_1d(o2)

    # enforce equal shapes (no broadcasting surprises)
    if not (pt.shape == sa.shape == ro.shape == o2.shape):
        raise ValueError(f"Inputs must have the same shape; got {pt.shape}, {sa.shape}, {ro.shape}, {o2.shape}")

    # elementwise validity mask
    valid = numpy.isfinite(pt) & numpy.isfinite(sa) & numpy.isfinite(ro) & numpy.isfinite(o2)

    out = numpy.full(pt.shape, numpy.nan, dtype=float)

    # constants.  <-- KM: cm3/dm3 constants
    A0 = 2.00907
    A1 = 3.22014
    A2 = 4.05010
    A3 = 4.94457
    A4 = -0.256847
    A5 = 3.88767
    B0 = -6.24523e-3
    B1 = -7.37614e-3
    B2 = -1.03410e-2
    B3 = -8.17083e-3
    C0 = -4.88682e-7

    # compute only on valid positions
    ptv, sav, rov, o2v = pt[valid], sa[valid], ro[valid], o2[valid]

    Ts = numpy.log((298.15 - ptv) / (273.15 + ptv))
    l  = (A0
          + A1*Ts + A2*(Ts**2) + A3*(Ts**3) + A4*(Ts**4) + A5*(Ts**5)
          + sav*(B0 + B1*Ts + B2*(Ts**2) + B3*(Ts**3))
          + C0*(sav**2))

    O2_eq_mmol_per_m3 = (1000/22.3916) * numpy.exp(l)    # <-- KM: here we're converting using the molar volume of O2, but this I guess has some assumptions in it - STP?
    O2_eq_umol_per_kg = O2_eq_mmol_per_m3 * rov / 1000
    out[valid] = O2_eq_umol_per_kg - o2v

    if scalar:
        return float(out[0])
    return out

In [4]:
# gsw implementation
def AOU_estimate_gsw(SA, CT, p, lon, lat, oxygen):

    o2sol = gsw.O2sol(SA, CT, p, lon, lat)
    O2_eq_umol_per_kg = o2sol * gsw.rho(SA, CT, p) / 1000

    return O2_eq_umol_per_kg - oxygen

In [6]:
# example based off https://www.teos-10.org/pubs/gsw/html/gsw_O2sol.html:

SA = [34.7118, 34.8915, 35.0256, 34.8472, 34.7366, 34.7324]
CT = [28.8099, 28.4392, 22.7862, 10.2262, 6.8272, 4.3236]
p =  [10, 50, 125, 250, 600, 1000]
lat =  [4, 4, 4, 4, 4, 4]
long = [188, 188, 188, 188, 188, 188]
potential_temperature = gsw.pt_from_CT(SA, CT)
salinity = gsw.SP_from_SA(SA, p, long, lat)
density = gsw.rho(SA, CT, p)
oxygen = [0,0,0,0,0,0]

adhoc_AOU = AOU_estimate(potential_temperature, salinity, density, oxygen)
gsw_AOU = AOU_estimate_gsw(SA, CT, p, long, lat, oxygen)

In [7]:
print(adhoc_AOU)

[203.5708565  204.65385753 225.40507747 288.99983357 312.60233691
 332.28098609]


In [8]:
print(gsw_AOU)

[199.22315737 200.23354249 220.141552   281.47631455 304.32843056
 323.38669933]
